In [ ]:
import sys
sys.path.append('/Users/nf/Documents/Projects/SkyLake')

from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, NumericType, StructField, StructType, StringType, IntegerType

from spark import config

In [ ]:
session = config.Session()

In [ ]:
spark = session.get_spark_session()

In [ ]:
flight_df = spark.read.csv(session.RAW_PATH, header=True)

In [7]:
flight_df.select("FL_DATE", "OP_CARRIER", "ORIGIN", "DEST", "DEP_DELAY", "ARR_DELAY").show(5)

+----------+----------+------+----+---------+---------+
|   FL_DATE|OP_CARRIER|ORIGIN|DEST|DEP_DELAY|ARR_DELAY|
+----------+----------+------+----+---------+---------+
|2018-01-01|        UA|   EWR| DEN|     -5.0|    -23.0|
|2018-01-01|        UA|   LAS| SFO|     -8.0|    -24.0|
|2018-01-01|        UA|   SNA| DEN|     -5.0|    -13.0|
|2018-01-01|        UA|   RSW| ORD|      6.0|     -2.0|
|2018-01-01|        UA|   ORD| ALB|     20.0|     14.0|
+----------+----------+------+----+---------+---------+
only showing top 5 rows


In [8]:
flight_df.count()

7213446

In [9]:
flight_df.printSchema()

root
 |-- FL_DATE: string (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: string (nullable = true)
 |-- TAXI_OUT: string (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: string (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- ARR_DELAY: string (nullable = true)
 |-- CANCELLED: string (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: string (nullable = true)
 |-- CRS_ELAPSED_TIME: string (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: string (nullable = true)
 |-- AIR_TIME: string (nullable = true)
 |-- DISTANCE: string (nullable = true)
 |-- CARRIER_DELAY: string (nullable = true)
 |-- WEATHER_DELAY: strin

In [10]:
flight_df.describe().show()

26/08/01 12:46:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 5:==========================================>                (5 + 2) / 7]

+-------+----------+----------+------------------+-------+-------+------------------+------------------+-----------------+------------------+------------------+-----------------+------------------+-----------------+-----------------+-----------------+-------------------+-----------------+--------------------+------------------+-------------------+------------------+-----------------+-----------------+------------------+------------------+-------------------+-------------------+-----------+
|summary|   FL_DATE|OP_CARRIER| OP_CARRIER_FL_NUM| ORIGIN|   DEST|      CRS_DEP_TIME|          DEP_TIME|        DEP_DELAY|          TAXI_OUT|        WHEELS_OFF|        WHEELS_ON|           TAXI_IN|     CRS_ARR_TIME|         ARR_TIME|        ARR_DELAY|          CANCELLED|CANCELLATION_CODE|            DIVERTED|  CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|          AIR_TIME|         DISTANCE|    CARRIER_DELAY|     WEATHER_DELAY|         NAS_DELAY|     SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|
+-------+-

In [11]:
flight_df = flight_df.drop("Unnamed: 27")

In [12]:
len(flight_df.columns)

27

In [13]:
flight_df.columns

['FL_DATE',
 'OP_CARRIER',
 'OP_CARRIER_FL_NUM',
 'ORIGIN',
 'DEST',
 'CRS_DEP_TIME',
 'DEP_TIME',
 'DEP_DELAY',
 'TAXI_OUT',
 'WHEELS_OFF',
 'WHEELS_ON',
 'TAXI_IN',
 'CRS_ARR_TIME',
 'ARR_TIME',
 'ARR_DELAY',
 'CANCELLED',
 'CANCELLATION_CODE',
 'DIVERTED',
 'CRS_ELAPSED_TIME',
 'ACTUAL_ELAPSED_TIME',
 'AIR_TIME',
 'DISTANCE',
 'CARRIER_DELAY',
 'WEATHER_DELAY',
 'NAS_DELAY',
 'SECURITY_DELAY',
 'LATE_AIRCRAFT_DELAY']

In [14]:
flight_df = flight_df.withColumn('FL_DATE', F.to_date(flight_df['FL_DATE'], 'yyyy-MM-dd'))

In [15]:
flight_df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: string (nullable = true)
 |-- TAXI_OUT: string (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: string (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- ARR_DELAY: string (nullable = true)
 |-- CANCELLED: string (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: string (nullable = true)
 |-- CRS_ELAPSED_TIME: string (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: string (nullable = true)
 |-- AIR_TIME: string (nullable = true)
 |-- DISTANCE: string (nullable = true)
 |-- CARRIER_DELAY: string (nullable = true)
 |-- WEATHER_DELAY: string 

In [16]:
double_cols = ["DEP_DELAY", "ARR_DELAY", "DISTANCE", "TAXI_OUT", "TAXI_IN", "CRS_ELAPSED_TIME", "ACTUAL_ELAPSED_TIME", "CARRIER_DELAY", "WEATHER_DELAY",
                 "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY", "AIR_TIME"]

for col in double_cols:
    flight_df = flight_df.withColumn(col, F.col(col).cast(DoubleType()))


flight_df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: string (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: string (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double 

In [17]:
flight_df = flight_df.withColumn("CANCELLED", F.col("CANCELLED").cast(DoubleType()).cast(IntegerType()))
flight_df = flight_df.withColumn("DIVERTED", F.col("DIVERTED").cast(DoubleType()).cast(IntegerType()))

In [18]:
flight_df = flight_df.fillna(0.0, subset=["CARRIER_DELAY", "WEATHER_DELAY",
                 "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"])

In [19]:
flight_df.select("FL_DATE", "OP_CARRIER", "ORIGIN", "DEST",
                 "CANCELLED", "DIVERTED"
                ).show(5)

+----------+----------+------+----+---------+--------+
|   FL_DATE|OP_CARRIER|ORIGIN|DEST|CANCELLED|DIVERTED|
+----------+----------+------+----+---------+--------+
|2018-01-01|        UA|   EWR| DEN|        0|       0|
|2018-01-01|        UA|   LAS| SFO|        0|       0|
|2018-01-01|        UA|   SNA| DEN|        0|       0|
|2018-01-01|        UA|   RSW| ORD|        0|       0|
|2018-01-01|        UA|   ORD| ALB|        0|       0|
+----------+----------+------+----+---------+--------+
only showing top 5 rows


In [20]:
flight_df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: string (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: string (nullable = true)
 |-- DEP_TIME: string (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: string (nullable = true)
 |-- WHEELS_ON: string (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: string (nullable = true)
 |-- ARR_TIME: string (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: integer (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: integer (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = false)
 |-- WEATHER_DELAY: doub

In [ ]:
flight_df.write.mode('overwrite').parquet(session.PROCESSED_PATH)